# Kestrel student: QLoRA fine-tune (Kaggle)

Runs `train.py` from the **kestrel-distill-code** dataset (the zip `scripts/build_train_snapshot.py` builds, with
its `COMMIT`): first the GPU smoke test on the synthetic
`train.jsonl` that sits next to it, then (only if you set `RUN_TRAINING = True`) the real training on the
**kestrel-distill-data** dataset (`train.jsonl`, `val.jsonl`, `manifest.json` from `kestrel distill build`).
The first cell finds both datasets under `/kaggle/input` by itself. All settings are in `train_config.toml`;
nothing is set here. Steps, expected run time and Kaggle gotchas: `distill/TRAINING.md`.

Settings needed: **Accelerator: GPU T4** (one is used), **Internet: on** (pip and the model download; the
switch needs a verified phone number). No token is needed (the base model isn't gated). Never paste a token
into a cell: use Add-ons > Secrets.

In [ ]:
# Finds the uploaded datasets under /kaggle/input: Kaggle mounts them at /kaggle/input/<slug>/ or (newer)
# /kaggle/input/datasets/<user>/<slug>/. Files sit flat in a dataset, with no subfolders.
import os
import subprocess
import sys
from pathlib import Path

INPUT = Path("/kaggle/input")
OUT = "/kaggle/working/run"  # outputs only under /kaggle/working: /kaggle/input is read-only
SMOKE_OUT = "/kaggle/working/gpu-smoke"
CODE_FILES = ["train.py", "train_format.py", "train_config.toml", "train.jsonl", "COMMIT"]  # the snapshot zip's files
DATA_FILES = ["train.jsonl", "val.jsonl", "manifest.json"]


def folders_with(root, names):
    """Folders under root that hold every file in names."""
    return sorted(Path(d) for d, _, files in os.walk(root, followlinks=True) if set(names) <= set(files))


def sh(*args):
    """Runs a command, streaming its output into the cell; stops the notebook if it fails."""
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"{Path(args[1]).name} exited with {p.returncode}: see the output above")


code = folders_with(INPUT, CODE_FILES)
if len(code) != 1:
    raise RuntimeError(
        f"expected one folder under {INPUT} with {', '.join(CODE_FILES)}; found {[str(p) for p in code] or 'none'}. "
        "Add the code dataset (Input > Add Input): the zip from scripts/build_train_snapshot.py, files flat."
    )
CODE = str(code[0])
data = [p for p in folders_with(INPUT, DATA_FILES) if str(p) != CODE]  # the smoke set is never training data
if len(data) > 1:
    raise RuntimeError(f"more than one training set under {INPUT}: {[str(p) for p in data]}; attach only one")
DATA = str(data[0]) if data else None
print(f"CODE (train.py and the smoke train.jsonl): {CODE}")
print(f"Code commit: {Path(CODE, 'COMMIT').read_text().strip()} (lineage.json records it)")
print(f"DATA (training set): {DATA or 'none attached (fine for the GPU smoke test)'}")
print(f"Outputs: {SMOKE_OUT}, {OUT}")

In [ ]:
print(sys.version)
assert sys.version_info >= (3, 11), "trl needs Python 3.11+"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no GPU: set Accelerator to GPU T4")

In [ ]:
# Installs the versions Kestrel's uv.lock pins (Kaggle's preinstalled CUDA build of torch is kept), then stops
# if any library isn't at its pin, so nothing runs on Kaggle's preinstalled versions. Pip's "dependency conflict"
# lines about Kaggle's own packages (e.g. bigframes) are harmless: train.py doesn't use them.
import importlib
import socket
from importlib import metadata

REQUIREMENTS = [
    "transformers==5.20.1",
    "peft==0.21.2",
    "trl==1.16.0",
    "datasets==5.1.0",
    "accelerate==1.15.0",
    "bitsandbytes==0.50.2",
]
for host in ("pypi.org", "huggingface.co"):
    try:
        socket.getaddrinfo(host, 443)
    except OSError as e:
        raise RuntimeError(
            f"No internet (can't resolve {host}: {e}). Turn on Settings > Internet; the switch needs a verified "
            "phone number (kaggle.com > your picture > Settings > Phone verification). The session restarts: "
            "run the notebook again from the top."
        ) from None
sh(sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS)

importlib.invalidate_caches()
wrong = []
for requirement in REQUIREMENTS:
    name, pin = requirement.split("==")
    try:
        found = metadata.version(name)
    except metadata.PackageNotFoundError:
        found = "missing"
    print(f"{name:13} {found:12} pin {pin:10} {'ok' if found == pin else 'MISMATCH'}")
    if found != pin:
        wrong.append(f"{name} {found} (pin {pin})")
print(f"{'torch':13} {metadata.version('torch'):12} Kaggle's build, not pinned")
if wrong:
    raise RuntimeError(f"not at the pinned version: {', '.join(wrong)}. Stopping before any GPU time is spent.")
# train.py runs in a new process, so pip's "restart the kernel" note doesn't apply to it.

In [ ]:
# Optional: a Hugging Face token only raises download rate limits. Add it as a Kaggle secret named HF_TOKEN
# (Add-ons > Secrets, then tick it for this notebook); it is read here and never printed or saved.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found")
except Exception:
    print("No HF_TOKEN secret (fine: the model isn't gated)")

## GPU smoke test: run this first

The real base model in 4-bit overfits a fixed batch of the synthetic examples (`train.jsonl` next to
`train.py` in the code dataset): the loss must fall by half, every example must have trainable tokens, and a
greedy generation must make a tool call that train_format parses. It also prints the library versions next to
the pins. A few minutes; the notebook stops here if it says FAILED.

In [ ]:
sh(
    sys.executable,
    f"{CODE}/train.py",
    "--gpu-smoke",
    "--data",
    CODE,
    "--out",
    SMOKE_OUT,
    "--config",
    f"{CODE}/train_config.toml",
)

## Full training (skipped unless `RUN_TRAINING = True`)

Runs only after the GPU smoke test PASSED and the **kestrel-distill-data** dataset is attached: set
`RUN_TRAINING = True` in the next cell. It checks that `train.jsonl`, `val.jsonl` and `manifest.json` are all
there before the model is loaded. 30-45 minutes on one T4 (`distill/TRAINING.md`, step 5).

In [ ]:
RUN_TRAINING = False  # set to True for the real run, after the GPU smoke test PASSED

if not RUN_TRAINING:
    print("Full training skipped: set RUN_TRAINING = True in this cell to run it.")
else:
    if DATA is None:
        raise RuntimeError(f"no training set under {INPUT}: attach kestrel-distill-data ({', '.join(DATA_FILES)})")
    missing = [name for name in DATA_FILES if not Path(DATA, name).is_file()]
    if missing:
        raise RuntimeError(f"{DATA} is missing {', '.join(missing)}: upload the whole `kestrel distill build` folder")
    sh(sys.executable, f"{CODE}/train.py", "--data", DATA, "--out", OUT, "--config", f"{CODE}/train_config.toml")

In [ ]:
# Package the adapter and its lineage for download (Output tab > kestrel-adapter.zip). Checkpoints stay behind.
import json
import shutil

if not RUN_TRAINING:
    print("Nothing to package: the full training was skipped.")
else:
    lineage = json.loads(Path(OUT, "lineage.json").read_text())
    print(json.dumps(lineage["results"] | {"dataset": lineage["dataset"]["manifest_sha256"]}, indent=2, default=str))
    package = Path("/kaggle/working/kestrel-adapter")
    shutil.rmtree(package, ignore_errors=True)
    shutil.copytree(Path(OUT, "adapter"), package / "adapter")
    shutil.copy(Path(OUT, "lineage.json"), package / "lineage.json")
    print(shutil.make_archive("/kaggle/working/kestrel-adapter", "zip", package))